In [1]:
import os
import logging
import matplotlib
matplotlib.use('Agg')  # save to file instead of opening an interactive window
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sqlalchemy
import xgboost as xgb
import joblib
from dotenv import load_dotenv, find_dotenv
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ==========================================
# 0. LOGGING & CONFIG
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("ScoutAI")

CONFIG = {
    "source_view": "view_scout_master",
    "predictions_table": "player_predictions",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "xgb_params": {
        "n_estimators": 1200,
        "learning_rate": 0.03,
        "max_depth": 7,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
    },
    "wonderkid_age_threshold": 22,
    "passport_tier_1_max_rank": 15,
    "passport_tier_2_max_rank": 60,
    "league_weights": {
        "Premier League": 1.5,
        "LaLiga": 1.4,
        "Serie A": 1.3,
        "Bundesliga": 1.3,
        "Ligue 1": 1.2,
        "Eredivisie": 1.15,
        "Liga Portugal": 1.15,
        "Süper Lig": 1.05,
    },
}

NUMERIC_FEATURES = [
    "height_in_cm", "minutes_per_match", "total_goals", "total_assists",
    "goals_per_90", "assists_per_90", "total_yellow_cards", "total_red_cards",
    "international_goals", "club_squad_size", "club_avg_age", "stadium_seats",
    "foreigners_percentage", "contract_months_remaining", "max_career_transfer_fee",
    "most_recent_transfer_fee", "has_transfer_history",
]

PERFORMANCE_FEATURES = [
    "age", "height_in_cm", "total_appearances", "minutes_per_match",
    "total_goals", "total_assists", "goals_per_90", "assists_per_90",
    "total_yellow_cards", "total_red_cards", "international_caps", "international_goals",
    "club_squad_size", "club_avg_age", "stadium_seats", "foreigners_percentage",
    "wonderkid_hype", "league_coefficient",
]

MARKET_SIGNAL_FEATURES = [
    "has_transfer_history",
    "max_career_transfer_fee",
    "most_recent_transfer_fee",
    "contract_months_remaining",
]

CATEGORICAL_FEATURES = ["detailed_position", "foot", "passport_tier"]

# ==========================================
# DIRECTORY SETUP (Models, Images, Data)
# ==========================================
MODELS_DIR = "models"
IMAGES_DIR = "images"
DATA_DIR = "data"

for directory in [MODELS_DIR, IMAGES_DIR, DATA_DIR]:
    os.makedirs(directory, exist_ok=True)


# ==========================================
# 1. DATABASE CONNECTION & DATA RETRIEVAL
# ==========================================

def load_data(engine: sqlalchemy.engine.Engine) -> pd.DataFrame:
    """Load the pre-joined player feature view from Postgres."""
    logger.info("Connecting to the database...")
    df = pd.read_sql(f"SELECT * FROM {CONFIG['source_view']}", engine)
    logger.info(f"Successfully retrieved {len(df)} player records.")
    return df


# ==========================================
# 2. FEATURE ENGINEERING & DATA CLEANING
# ==========================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create derived features and coerce numeric columns."""
    df = df.copy()

    df["log_current_market_value"] = np.log1p(df["current_market_value"].astype(float))

    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["total_appearances"] = pd.to_numeric(df["total_appearances"], errors="coerce")
    df["international_caps"] = pd.to_numeric(df["international_caps"], errors="coerce")
    df["passport_power_rank"] = pd.to_numeric(df["passport_power_rank"], errors="coerce")

    df["wonderkid_hype"] = np.where(
        df["age"] <= CONFIG["wonderkid_age_threshold"],
        df["total_appearances"].fillna(0) + (df["international_caps"].fillna(0) * 3),
        0,
    )

    conditions = [
        df["passport_power_rank"].fillna(999) <= CONFIG["passport_tier_1_max_rank"],
        (df["passport_power_rank"].fillna(999) > CONFIG["passport_tier_1_max_rank"])
        & (df["passport_power_rank"].fillna(999) <= CONFIG["passport_tier_2_max_rank"]),
    ]
    df["passport_tier"] = np.select(conditions, ["Tier_1", "Tier_2"], default="Tier_3")

    df["league_coefficient"] = df["competition_name"].map(CONFIG["league_weights"]).fillna(1.0)
    df["detailed_position"] = df["sub_position"].fillna(df["position_group"]).astype(str)

    for col in NUMERIC_FEATURES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


# ==========================================
# 3. MODEL TRAINING
# ==========================================

def build_pipeline(existing_categorical: list) -> Pipeline:
    """Build the preprocessing + XGBoost regression pipeline."""
    preprocessor = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), existing_categorical)],
        remainder="passthrough",
    )
    xgb_model = xgb.XGBRegressor(**CONFIG["xgb_params"])
    return Pipeline(steps=[("preprocessor", preprocessor), ("regressor", xgb_model)])


def train_model(df: pd.DataFrame, use_market_signals: bool, model_label: str):
    """Run cross-validation, train/test split, and persist model to disk."""
    feature_pool = list(PERFORMANCE_FEATURES)
    if use_market_signals:
        feature_pool += MARKET_SIGNAL_FEATURES

    existing_features = [f for f in feature_pool if f in df.columns]
    existing_categorical = [f for f in CATEGORICAL_FEATURES if f in df.columns]

    X = df[existing_features + existing_categorical]
    y = df["log_current_market_value"]

    model_pipeline = build_pipeline(existing_categorical)

    logger.info(f"[{model_label}] Running {CONFIG['cv_folds']}-fold cross-validation...")
    kfold = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
    cv_scores = cross_val_score(
        model_pipeline, X, y, cv=kfold, scoring="r2", n_jobs=-1
    )
    logger.info(
        f"[{model_label}] CV R² mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f}) "
        f"| folds: {np.round(cv_scores, 4).tolist()}"
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CONFIG["test_size"], random_state=CONFIG["random_state"]
    )

    logger.info(f"[{model_label}] Training the final model...")
    model_pipeline.fit(X_train, y_train)
    test_r2 = model_pipeline.score(X_test, y_test)
    logger.info(f"[{model_label}] 🏆 R^2 SCORE (holdout test): {test_r2:.4f}")

    # Save model (.pkl) to the MODELS_DIR
    model_path = os.path.join(MODELS_DIR, f"scout_model_{model_label}.pkl")
    joblib.dump(model_pipeline, model_path)
    logger.info(f"[{model_label}] Model saved to '{model_path}'.")

    return model_pipeline, X, existing_features, existing_categorical, cv_scores, test_r2


# ==========================================
# 4. FEATURE IMPORTANCE VISUALIZATION
# ==========================================

def plot_feature_importance(model_pipeline: Pipeline, existing_features: list,
                             existing_categorical: list, model_label: str):
    """Save a horizontal bar chart of the top 20 most important features."""
    fitted_xgb = model_pipeline.named_steps["regressor"]
    cat_features_out = (
        model_pipeline.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .get_feature_names_out(existing_categorical)
    )
    all_features = list(cat_features_out) + existing_features
    importances = fitted_xgb.feature_importances_

    fi_df = pd.DataFrame({"Feature": all_features, "Importance": importances})
    fi_df["Feature"] = fi_df["Feature"].str.replace("_", " ").str.title()
    fi_df = fi_df.sort_values(by="Importance", ascending=False).head(20)

    plt.figure(figsize=(10, 8))
    plt.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1], color="#2c3e50")
    plt.title(f"Scout AI ({model_label}): Key Drivers of Market Value", fontsize=16)
    plt.tight_layout()
    
    # Save plot (.png) to the IMAGES_DIR
    plot_path = os.path.join(IMAGES_DIR, f"feature_importance_{model_label}.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    logger.info(f"[{model_label}] Feature importance plot saved to '{plot_path}'.")


# ==========================================
# 5. FINAL PREDICTION, OPPORTUNITY SCORE & EXPORT
# ==========================================

def run_predictions(df: pd.DataFrame, model_pipeline: Pipeline, X: pd.DataFrame,
                     value_col: str, gap_col: str, gap_pct_col: str) -> pd.DataFrame:
    """Predict market value for every player and compute the opportunity gap."""
    logger.info("Running global market value predictions...")
    predictions = model_pipeline.predict(X)
    df = df.copy()
    df[value_col] = np.expm1(predictions)

    df[gap_col] = df[value_col] - df["current_market_value"]
    df[gap_pct_col] = (df[gap_col] / df["current_market_value"].replace(0, np.nan)) * 100

    return df


def save_predictions_to_db(df_preds: pd.DataFrame, engine: sqlalchemy.engine.Engine, table_name: str):
    """Write predictions back to Postgres."""
    df_preds.to_sql(table_name, engine, if_exists="replace", index=False)
    logger.info(f"Predictions successfully saved to '{table_name}' table.")


def export_predictions_csv(df: pd.DataFrame, columns: list, csv_path: str, sort_col: str):
    """Export a ranked CSV of predictions."""
    export_cols = [c for c in columns if c in df.columns]
    df[export_cols].sort_values(sort_col, ascending=False).to_csv(csv_path, index=False)
    logger.info(f"Predictions exported to '{csv_path}'.")


# ==========================================
# MAIN
# ==========================================

def run_pipeline_for_model(df: pd.DataFrame, engine: sqlalchemy.engine.Engine,
                            use_market_signals: bool, model_label: str):
    """Train, evaluate, predict, and export for a single model variant."""
    model_pipeline, X, existing_features, existing_categorical, cv_scores, test_r2 = train_model(
        df, use_market_signals=use_market_signals, model_label=model_label
    )

    plot_feature_importance(model_pipeline, existing_features, existing_categorical, model_label)

    value_col = f"predicted_value_{model_label}"
    gap_col = f"value_gap_{model_label}"
    gap_pct_col = f"value_gap_pct_{model_label}"

    df_pred = run_predictions(
        df, model_pipeline, X[existing_features + existing_categorical],
        value_col=value_col, gap_col=gap_col, gap_pct_col=gap_pct_col,
    )

    predictions_df = df_pred[["player_id", value_col, gap_col, gap_pct_col]]
    save_predictions_to_db(predictions_df, engine, table_name=f"player_predictions_{model_label}")

    # Save CSV to the DATA_DIR
    csv_path = os.path.join(DATA_DIR, f"scout_predictions_export_{model_label}.csv")
    export_predictions_csv(
        df_pred,
        columns=["player_id", "player_name", "club_name", "competition_name", "age",
                 "current_market_value", value_col, gap_col, gap_pct_col],
        csv_path=csv_path,
        sort_col=gap_pct_col,
    )

    return df_pred, cv_scores, test_r2


def main():
    db_url = os.getenv('DB_URL')
    if not db_url:
        raise ValueError("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")

    engine = sqlalchemy.create_engine(db_url)

    df = load_data(engine)
    df = engineer_features(df)

    df_full, cv_full, r2_full = run_pipeline_for_model(
        df, engine, use_market_signals=True, model_label="full"
    )

    df_perf, cv_perf, r2_perf = run_pipeline_for_model(
        df, engine, use_market_signals=False, model_label="performance_only"
    )

    logger.info("=" * 60)
    logger.info("SUMMARY")
    logger.info(f"Full model            -> CV R²: {cv_full.mean():.4f}  | Holdout R²: {r2_full:.4f}")
    logger.info(f"Performance-only model -> CV R²: {cv_perf.mean():.4f}  | Holdout R²: {r2_perf:.4f}")
    logger.info("=" * 60)
    logger.info("Scout AI Opportunity Mode is ready.")


if __name__ == "__main__":
    main()

[09:13:16] INFO: Connecting to the database...
[09:13:19] INFO: Successfully retrieved 20380 player records.
[09:13:19] INFO: [full] Running 5-fold cross-validation...
[09:13:42] INFO: [full] CV R² mean: 0.7525 (+/- 0.0074) | folds: [0.7517, 0.7603, 0.7503, 0.76, 0.7401]
[09:13:42] INFO: [full] Training the final model...
[09:13:49] INFO: [full] 🏆 R^2 SCORE (holdout test): 0.7483
[09:13:49] INFO: [full] Model saved to 'models\scout_model_full.pkl'.
[09:13:49] INFO: [full] Feature importance plot saved to 'images\feature_importance_full.png'.
[09:13:49] INFO: Running global market value predictions...
[09:13:50] INFO: Predictions successfully saved to 'player_predictions_full' table.
[09:13:50] INFO: Predictions exported to 'data\scout_predictions_export_full.csv'.
[09:13:50] INFO: [performance_only] Running 5-fold cross-validation...
[09:14:11] INFO: [performance_only] CV R² mean: 0.7158 (+/- 0.0057) | folds: [0.7117, 0.7193, 0.718, 0.7229, 0.707]
[09:14:11] INFO: [performance_only] Tr